# 05 · Do time and location help us suggest tags?

**DR-19 question:** Does adding the context already in our export improve the word TF-IDF classifier from Notebook 04?

Compare **F0: the original title** with **F3: the same title plus weekday, time bucket, venue tokens and postcode**. Both models use the same training examples, classifier settings and validation division. This is a small data-science experiment; there is no service or API to build here.

The filename reserves room for nearby places, but this round uses only existing fields. It does not look up coordinates, embassies or other places. F4 nearby-place enrichment remains a separate stretch task. No LLM or paid API is needed.

**What would a gain mean?** Context might help identify topics associated with a venue or schedule. It can also teach a shortcut that fails when an event moves. The event-group split keeps related editions together, but venues can occur in several groups. This is not a test of generalization to new venues or future dates. A location alone is not proof of a topic.

All model fitting uses training records. We choose each cutoff on 94 validation examples and check it on another 94. The 40 reviewed pilot records are familiar development data. The check data have already been inspected in earlier experiments, and historical tags may be wrong. These results are exploratory; the final-test answers are not used for fitting, tuning or scoring.

Run with the **DemoLogic (uv)** kernel and the locked `notebooks` dependency group. The notebook explains the features, compares results, shows changed suggestions, and saves a private report. Clear outputs before committing.

In [ ]:
import platform
import re
import sys
from datetime import UTC, datetime
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer

root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None
)
if root is None:
    raise FileNotFoundError("Open this notebook inside the DemoLogic repository")
sys.path.insert(0, str(root / "src"))
sys.path.insert(0, str(root))

from demo_logic.benchmark import _csv, load_benchmark, representative_selection
from demo_logic.evaluation import multilabel_metrics, parse_review_tags
from demo_logic.notebook_support import save_private_json
from demo_logic.snapshot import load_snapshot
from demo_logic.taxonomy import topic_tags
from notebooks.tools import (
    aligned_predictions,
    comparison_row,
    file_sha256,
    fingerprint,
    matching_report,
    notebook_sha256,
    percent,
)
from notebooks.tools import show_table as show

## 1. Keep the same training and validation examples

Load the verified snapshot and saved group split, then retain one representative per event type using DR-40's selection. We do not select easier examples based on whether location is present. The benchmark loader checks membership, including the final-review file; those answers do not enter this experiment. Only training and validation representatives reach feature construction.

In [ ]:
rows, source = load_snapshot(root / "data/dataset.jsonl", root / "data/manifest.json")
bundle = load_benchmark(root / "data/benchmark", rows, source["content_sha256"])
if bundle.manifest.get("status") != "frozen" or source["taxonomy_version"] != "v2":
    raise ValueError("Notebook 05 requires locked split membership and taxonomy v2")

by_id = {row["external_id"]: row for row in rows}
development = {name: [] for name in ("training", "validation")}
for assignment in bundle.manifest["assignments"]:
    if assignment["split"] in development:
        development[assignment["split"]].append(
            {**by_id[assignment["external_id"]], "group_id": assignment["group_id"]}
        )
selection = representative_selection(bundle)
selection_path = save_private_json(
    root / "data/benchmark" / f"{selection['version']}.json", selection
)
selected_ids = {item["external_id"] for item in selection["records"]}
# Keep full validation history only for the separate repeat-event simulation.
validation_occurrences = development["validation"]
training, validation = (
    [row for row in development[name] if row["external_id"] in selected_ids]
    for name in ("training", "validation")
)
assert set(bundle.pilot_ids).issubset({row["external_id"] for row in validation})
tags = sorted(topic_tags(source["taxonomy_version"]))
reviewed_pilot = {
    row["external_id"]: parse_review_tags(row["reviewed_tags"], set(tags))
    for row in _csv(bundle.paths["rubric_pilot"], {"external_id", "reviewed_tags"})
}
if len(reviewed_pilot) != 40:
    raise ValueError("The 40-record development pilot must be complete")

show(
    ["Input", "Count or version"],
    [
        ("Historical training records", len(training)),
        ("Historical validation records", len(validation)),
        ("Reviewed development pilot", len(reviewed_pilot)),
        ("Tag catalog", source["taxonomy_version"]),
        ("Final-test records used for fitting or scoring", 0),
    ],
)

## 2. Turn the extra fields into words the classifier can read

TF-IDF reads tokens. We append named tokens to the title, such as `ctx_weekday_0`, `ctx_time_09_00`, `ctx_postcode_10117` and `ctx_venue_beispielplatz`. Monday is 0 and Sunday is 6. Time buckets come from the prepared export; we do not calculate a new bucket here. The `ctx_` prefix distinguishes context from ordinary title words. Underscores keep each token together in the word vectorizer.

A synthetic event might have the title **Sichere Radwege jetzt**, Monday morning, and an example venue. F0 sees only the title. F3 also sees its context tokens, as shown below. The classifier learns associations from training data; we do not tell it that a postcode means a particular tag.

Missing fields get explicit `ctx_*_missing` tokens. A missing venue does not remove the example. The same word-vectorizer settings apply to both variants: single words and word pairs, minimum two training documents, at most 30,000 features. Each vocabulary is learned on training text only. Adding context can change TF-IDF weights and create title/context word pairs; this is a comparison of this complete feature recipe, not a claim that location causes the event topic.

In [ ]:
def context_tokens(row):
    """Encode only the approved context fields as stable, namespaced word tokens.

    Keep Monday (0) and midnight as real values. Use explicit missing markers
    for empty fields. Sort and deduplicate venue tokens so their input order
    does not change the representation. Never read labels, IDs or review notes.
    """

    def token(prefix, value):
        text = "" if value is None else str(value).strip().lower()
        text = re.sub(r"[^\w]+", "_", text).strip("_")
        return f"ctx_{prefix}_{text or 'missing'}"

    values = [
        token("weekday", row.get("weekday")),
        token("time", row.get("time_bucket")),
        token("postcode", row.get("postcode")),
    ]
    venues = sorted(
        {token("venue", item) for item in (row.get("venue_tokens") or []) if str(item).strip()}
    )
    return values + (venues or ["ctx_venue_missing"])


def feature_text(row, variant):
    """Keep the original title; append existing context only for the declared F3 variant."""
    if variant == "F0":
        return row["title"]
    if variant == "F3":
        return row["title"] + "\n" + " ".join(context_tokens(row))
    raise ValueError("Expected F0 or F3")


synthetic = {
    "title": "Sichere Radwege jetzt",
    "weekday": 0,
    "time_bucket": "09:00",
    "postcode": "10117",
    "venue_tokens": ["beispielplatz"],
}
show(
    ["Input variant", "Text sent to TF-IDF"],
    [(name, feature_text(synthetic, name)) for name in ["F0", "F3"]],
)
missing_context = {}
for name, records in [("training", training), ("validation", validation)]:
    missing_context[name] = {
        field: sum(
            row.get(field) is None or row.get(field) == "" or row.get(field) == []
            for row in records
        )
        for field in ["weekday", "time_bucket", "postcode", "venue_tokens"]
    }
show(
    ["Split", "Records", "Missing weekday", "Missing time", "Missing postcode", "Missing venue"],
    [
        (name, len(records), *missing_context[name].values())
        for name, records in [("training", training), ("validation", validation)]
    ],
)
print("Summary: retain every selected event, including those with missing context.")

## 3. Train the same classifier twice

Use the word-only model from Notebook 04 as the fixed reference, rather than choosing a model after seeing this experiment. Each tag gets a separate logistic-regression classifier. The two runs differ only in their input text: F0 or F3. Fit TF-IDF on training records, then transform validation records without learning from them. Save timing for preparation, training and scoring separately. The scores are used for ranking and cutoff selection; they are not calibrated confidence percentages.

In [ ]:
RANDOM_SEED = 42
THRESHOLDS = [round(value, 2) for value in np.arange(0.05, 0.81, 0.05)]
WORD_SETTINGS = {
    "analyzer": "word",
    "ngram_range": (1, 2),
    "min_df": 2,
    "max_features": 30000,
    "sublinear_tf": True,
    "dtype": np.float64,
}
CLASSIFIER_SETTINGS = {"solver": "liblinear", "max_iter": 1000, "random_state": RANDOM_SEED}
encoder = MultiLabelBinarizer(classes=tags)
y_train = encoder.fit_transform([row["tag_slugs"] for row in training])
if np.any(y_train.sum(axis=0) == 0):
    raise ValueError("Every tag needs training support")
fits = {}
for variant in ["F0", "F3"]:
    started = perf_counter()
    train_text = [feature_text(row, variant) for row in training]
    valid_text = [feature_text(row, variant) for row in validation]
    preparation_seconds = perf_counter() - started
    vectorizer = TfidfVectorizer(**WORD_SETTINGS)
    started = perf_counter()
    x_train = vectorizer.fit_transform(train_text)
    train_vectorize_seconds = perf_counter() - started
    classifier = OneVsRestClassifier(LogisticRegression(**CLASSIFIER_SETTINGS))
    started = perf_counter()
    classifier.fit(x_train, y_train)
    fit_seconds = perf_counter() - started
    started = perf_counter()
    x_valid = vectorizer.transform(valid_text)
    probabilities = classifier.predict_proba(x_valid)
    prediction_seconds = perf_counter() - started
    if probabilities.shape != (len(validation), len(tags)) or not np.isfinite(probabilities).all():
        raise ValueError("Unexpected classifier scores")
    usable_context = set(vectorizer.get_feature_names_out())
    fits[variant] = {
        "probabilities": probabilities,
        "feature_count": x_train.shape[1],
        "preparation_seconds": preparation_seconds,
        "train_vectorize_seconds": train_vectorize_seconds,
        "fit_seconds": fit_seconds,
        "prediction_seconds": prediction_seconds,
        "input_sha256": fingerprint({"training": train_text, "validation": valid_text}),
        "context_feature_count": sum("ctx_" in token for token in usable_context),
        "validation_with_known_venue_token": sum(
            any(
                t in usable_context
                for t in context_tokens(row)
                if t.startswith("ctx_venue_") and t != "ctx_venue_missing"
            )
            for row in validation
        ),
    }
show(
    [
        "Variant",
        "Features",
        "Context features",
        "Prepare (s)",
        "Vectorize training (s)",
        "Fit (s)",
        "Predict 188 (s)",
    ],
    [
        (
            name,
            run["feature_count"],
            run["context_feature_count"],
            *[
                round(run[k], 3)
                for k in [
                    "preparation_seconds",
                    "train_vectorize_seconds",
                    "fit_seconds",
                    "prediction_seconds",
                ]
            ],
        )
        for name, run in fits.items()
    ],
)
print(
    "F3 validation events with at least one learned venue token:",
    fits["F3"]["validation_with_known_venue_token"],
)

## 4. Choose the cutoff, then check it on different examples

**First 94:** Try the same fixed cutoffs as Notebook 04 for each input variant. Keep the cutoff with the highest micro F1; if tied, keep the higher cutoff. **Other 94:** Apply that choice without adjusting it. The selected event groups are disjoint and match Notebook 04.

We report the usual two tags plus exact ties at second place, and the strict two-tag alternative. Precision asks how many suggested tags match our references; recall asks how many reference tags we found; coverage asks how many events receive any suggestion. The reference tags are historical assignments, so an apparent error may also be an old labeling problem.

In [ ]:
groups = [row["group_id"] for row in validation]
splitter = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=RANDOM_SEED)
tune_index, check_index = next(splitter.split([row["title"] for row in validation], groups=groups))
if {groups[index] for index in tune_index} & {groups[index] for index in check_index}:
    raise AssertionError("Related event types crossed the validation check split")


def select_tags(scores, threshold, *, include_ties=True):
    """Return the highest-scoring approved tags that meet a tested threshold.

    Args:
        scores: One classifier score per tag, in the catalog's fixed order.
        threshold: Minimum score chosen only from development records.
        include_ties: Keep exact ties at the second tag; otherwise cap at two.

    An empty result is an abstention. Slug order breaks equal-score ties so
    rerunning the notebook on the same data produces the same recommendation.
    """
    ranked = sorted(
        ((tag, float(score)) for tag, score in zip(tags, scores) if score >= threshold),
        key=lambda item: (-item[1], item[0]),
    )
    chosen = ranked[:2]
    if include_ties and len(ranked) > 2:
        chosen += [item for item in ranked[2:] if item[1] == chosen[-1][1]]
    return [tag for tag, _ in chosen]


def predictions_at(probabilities, threshold, include_ties):
    """Apply one explicit recommendation policy to every scored event."""
    return [select_tags(scores, threshold, include_ties=include_ties) for scores in probabilities]


def score_indices(reference, predictions, indices):
    """Measure the same event indices under a complete multi-label reference."""
    return multilabel_metrics(
        [reference[index] for index in indices],
        [predictions[index] for index in indices],
        tags,
    )


historical = [row["tag_slugs"] for row in validation]
policies = []
for name, run in fits.items():
    for include_ties in (True, False):
        tried = []
        for threshold in THRESHOLDS:
            predictions = predictions_at(run["probabilities"], threshold, include_ties)
            metrics = score_indices(historical, predictions, tune_index)
            tried.append((threshold, metrics))
        threshold, tune_metrics = max(
            tried,
            key=lambda item: (
                item[1]["micro_f1"] if item[1]["micro_f1"] is not None else -1,
                item[0],
            ),
        )
        predictions = predictions_at(run["probabilities"], threshold, include_ties)
        check_metrics = score_indices(historical, predictions, check_index)
        full_metrics = multilabel_metrics(historical, predictions, tags)
        pilot_indices = [
            index for index, row in enumerate(validation) if row["external_id"] in reviewed_pilot
        ]
        pilot_metrics = multilabel_metrics(
            [reviewed_pilot[validation[index]["external_id"]] for index in pilot_indices],
            [predictions[index] for index in pilot_indices],
            tags,
        )
        policies.append(
            {
                "model": name,
                "policy": "include_cutoff_ties" if include_ties else "strict_two",
                "threshold": threshold,
                "tuning": tune_metrics,
                "group_heldout_check": check_metrics,
                "historical_agreement": full_metrics,
                "reviewed_pilot": pilot_metrics,
                "predictions": predictions,
                "threshold_trials": [
                    {
                        "threshold": value,
                        "micro_precision": result["micro_precision"],
                        "micro_recall": result["micro_recall"],
                        "micro_f1": result["micro_f1"],
                        "coverage": result["coverage"],
                    }
                    for value, result in tried
                ],
            }
        )

show(
    ["Validation purpose", "Demonstrations (rows)", "Distinct event types"],
    [
        (label, len(indices), len({groups[index] for index in indices}))
        for label, indices in [
            ("Choose the cutoff", tune_index),
            ("Check the chosen cutoff", check_index),
        ]
    ],
)

show(
    ["Model", "Tag-limit policy", "Chosen cutoff"],
    [(p["model"], p["policy"], p["threshold"]) for p in policies],
)

## 5. Compare the results on the same events

Read precision, recall and coverage together. A model can find more tags by suggesting too many. The bars below use only the 94 check records; dashed lines mark the project targets for precision/recall and coverage. The pilot table uses 40 reviewed records that have already informed development, not an independent final test.

In [ ]:
columns = ["Variant / policy", "N", "Precision", "Recall", "F1", "Coverage", "Exact", ">2 tags"]
show(
    columns,
    [comparison_row(p["model"] + " / " + p["policy"], p["group_heldout_check"]) for p in policies],
)
print("Reviewed pilot (40 familiar development examples)")
show(
    columns,
    [comparison_row(p["model"] + " / " + p["policy"], p["reviewed_pilot"]) for p in policies],
)
main = {p["model"]: p for p in policies if p["policy"] == "include_cutoff_ties"}
metric_names = ["micro_precision", "micro_recall", "micro_f1", "coverage"]
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(4)
for offset, name in [(-0.18, "F0"), (0.18, "F3")]:
    ax.bar(
        x + offset,
        [main[name]["group_heldout_check"][k] or 0 for k in metric_names],
        width=0.36,
        label=name,
    )
ax.set(
    xticks=x,
    xticklabels=["Precision", "Recall", "F1", "Coverage"],
    ylim=(0, 1.05),
    ylabel="Fraction",
    title="Same 94 check events — agreement with historical tags",
)
ax.hlines(0.8, -0.45, 1.45, colors="gray", linestyles="dashed", label="Precision / recall target")
ax.hlines(0.95, 2.55, 3.45, colors="black", linestyles="dotted", label="Coverage target")
ax.legend()
fig.tight_layout()
plt.show()
plt.close(fig)
for name in ["F0", "F3"]:
    m = main[name]["group_heldout_check"]
    print(
        f"{name}: cutoff {main[name]['threshold']}; precision {percent(m['micro_precision'])}, recall {percent(m['micro_recall'])}, F1 {percent(m['micro_f1'])}, coverage {percent(m['coverage'])}"
    )
print(
    "F3 minus F0 F1, percentage points:",
    round(
        100
        * (
            main["F3"]["group_heldout_check"]["micro_f1"]
            - main["F0"]["group_heldout_check"]["micro_f1"]
        ),
        2,
    ),
)

expected_fingerprints = {
    str(selection_path.relative_to(root)): file_sha256(selection_path),
    "data/manifest.json": file_sha256(root / "data/manifest.json"),
    "data/benchmark/split-manifest.json": file_sha256(bundle.paths["manifest"]),
    "data/benchmark/rubric-pilot-queue.csv": file_sha256(bundle.paths["rubric_pilot"]),
}
base_path, base = matching_report(
    root / "reports/classical",
    "model_version",
    source=source,
    selection=selection,
    fingerprints=expected_fingerprints,
)
base_policy = next(
    p for p in base["policies"] if p["model"] == "word" and p["policy"] == "include_cutoff_ties"
)
if base["random_seed"] != RANDOM_SEED or base["threshold_grid"] != THRESHOLDS:
    raise ValueError("Notebook 04 used a different split seed or cutoff grid")
if aligned_predictions(base_policy["predictions"], validation) != main["F0"]["predictions"]:
    raise ValueError("F0 does not reproduce the saved Notebook 04 word-model predictions")
if base_policy["group_heldout_check"] != main["F0"]["group_heldout_check"]:
    raise ValueError("F0 metrics do not reproduce Notebook 04")
print("F0 reproduces the saved Notebook 04 predictions and check metrics.")

## 6. Inspect what changed

Compare each event's two prediction sets. The table shows up to eight changed cases, ordered by ID, not selected for looking successful. It includes the original title and added fields so we can question whether a venue or schedule helped or misled the model. A match against an old tag is not proof of correctness. Keep this row-level output private.

The summary counts exact matches gained and lost against historical labels. This is only one error diagnostic: partial improvements can occur even when neither prediction matches every reference tag. Per-tag precision and recall show which topics changed.

In [ ]:
changes = []
for i in sorted(check_index, key=lambda i: validation[i]["external_id"]):
    before, after = main["F0"]["predictions"][i], main["F3"]["predictions"][i]
    if set(before) == set(after):
        continue
    row = validation[i]
    changes.append(
        {
            "external_id": row["external_id"],
            "title": row["title"],
            "context": context_tokens(row),
            "historical": historical[i],
            "F0": before,
            "F3": after,
            "exact_before": set(before) == set(historical[i]),
            "exact_after": set(after) == set(historical[i]),
        }
    )
show(
    ["ID", "Title", "Added context", "Historical", "F0", "F3"],
    [
        (
            r["external_id"],
            r["title"],
            " ".join(r["context"]),
            ", ".join(r["historical"]),
            ", ".join(r["F0"]),
            ", ".join(r["F3"]),
        )
        for r in changes[:8]
    ],
)
change_summary = {
    "changed_events": len(changes),
    "exact_matches_gained": sum(not r["exact_before"] and r["exact_after"] for r in changes),
    "exact_matches_lost": sum(r["exact_before"] and not r["exact_after"] for r in changes),
}
show(["Change on 94 check examples", "Count"], change_summary.items())
show(
    ["Tag", "Reference count", "F0 precision", "F3 precision", "F0 recall", "F3 recall"],
    [
        (
            tag,
            main["F0"]["group_heldout_check"]["per_tag"][tag]["support"],
            *[
                percent(main[name]["group_heldout_check"]["per_tag"][tag][metric])
                for metric in ["precision", "recall"]
                for name in ["F0", "F3"]
            ],
        )
        for tag in tags
    ],
)

## 7. Save the experiment so we can compare it later

The report keeps feature settings, input fingerprints, the training/tuning/check IDs, every tested cutoff, scores, predictions, missing-field counts and the link to Notebook 04. It lives in the ignored `reports/context/` directory. No fitted artifact is selected or deployed automatically. The next cell writes a new report; earlier results remain available.

In [ ]:
run_id = datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
report = {
    "run_id": run_id,
    "experiment_version": "context-f0-f3-v1",
    "snapshot_sha256": source["content_sha256"],
    "selection_sha256": selection["selection_sha256"],
    "taxonomy_version": source["taxonomy_version"],
    "fingerprints": expected_fingerprints,
    "rubric_sha256": file_sha256(root / "docs/tagging-rubric.md"),
    "notebook_source_sha256": notebook_sha256(
        root / "notebooks/05_context_and_nearby_places.ipynb"
    ),
    "base_report": str(base_path.relative_to(root)),
    "base_report_sha256": file_sha256(base_path),
    "python": platform.python_version(),
    "sklearn": sklearn.__version__,
    "random_seed": RANDOM_SEED,
    "training_ids": [row["external_id"] for row in training],
    "threshold_selection_ids": [validation[i]["external_id"] for i in tune_index],
    "check_ids": [validation[i]["external_id"] for i in check_index],
    "threshold_grid": THRESHOLDS,
    "word_settings": {**WORD_SETTINGS, "dtype": "float64"},
    "classifier_settings": CLASSIFIER_SETTINGS,
    "context_fields": ["weekday", "time_bucket", "postcode", "venue_tokens"],
    "missing_context": missing_context,
    "change_summary": change_summary,
    "changed_predictions": changes,
    "resources": {
        name: {k: v for k, v in run.items() if k != "probabilities"} for name, run in fits.items()
    },
    "results": [
        {
            **{k: v for k, v in p.items() if k != "predictions"},
            "predictions": [
                {
                    "external_id": row["external_id"],
                    "predicted": prediction,
                    "scores": fits[p["model"]]["probabilities"][i].tolist(),
                }
                for i, (row, prediction) in enumerate(zip(validation, p["predictions"]))
            ],
        }
        for p in policies
    ],
    "score_tag_order": tags,
    "api_cost_usd": 0,
    "local_compute_cost_measured": False,
    "final_test_records_scored": 0,
    "limitations": [
        "Historical labels can be imperfect; validation has already been inspected",
        "Pilot is familiar development data",
        "Venue and time associations may be shortcuts",
        "Not a held-out-venue or future-date evaluation",
        "Single-run timings are not production latency",
        "No place enrichment; F1/F2 separation and F4 are not part of this run",
    ],
}
report_path = save_private_json(root / "reports/context" / f"{run_id}.json", report)
print("Private context report:", report_path)

## 8. Summary and next step — 29 September 2026

**Adding this context did not improve the overall check-set F1.** It made the classifier suggest tags for more events, but a smaller share of those tags matched the historical answers. Keep the title-only baseline; do not promote F3 from this experiment.

| Same 94 check events | Chosen cutoff | Precision | Recall | F1 | Coverage |
|---|---:|---:|---:|---:|---:|
| F0: original title | 0.25 | 55.3% | 49.6% | 52.3% | 81.9% |
| F3: title + time + location | 0.20 | 42.8% | 55.9% | 48.5% | 98.9% |

**What the chart tells us:** F3 reaches more events and finds more historical reference tags, but also adds more mismatches. F1 falls by 3.8 percentage points. Coverage exceeds our 95% target, while precision and recall remain below 80%. The different cutoffs were chosen on the tuning half; this compares the two complete methods, not added fields at one common cutoff. Strict-two and tie-inclusive policies give identical results in this run.

**The pilot gives a mixed signal:** On the familiar 40 reviewed examples, F1 rises from 39.3% to 42.2%. Precision falls from 48.0% to 41.3%, recall rises from 33.3% to 43.1%, and coverage rises from 80% to 100%. The reviewed pilot and historical check results are both development evidence. Neither establishes final-test quality or proves the historical labels are correct.

**Which predictions changed?** 51 of the 94 check events receive a different tag set. Four become exact matches to the historical answers, while eleven lose an exact match. Mobility F1 improves, but several other tags worsen; rare tags have too few examples for strong claims. These are differences from historical labels, not automatically confirmed model mistakes.

**What context was available?** Among the 188 validation records, 39 lack a postcode and 43 lack venue tokens; these missing fields can overlap. All have weekday and time fields. We retain every example. Only 100 validation records contain at least one venue token learned by the F3 vectorizer; this does not mean that 100 complete venues are known. The feature count rises from 2,098 to 2,684. Context may provide shortcuts tied to places and schedules, and its addition also changes TF-IDF weights.

**Decision for review:** Keep this bounded comparison as evidence. Do not add nearby-place research, separate time/location variants or an SVM in this round. The current result does not justify expanding context work ahead of the core hosted-model comparisons. It does not establish that all context representations are unhelpful. Notebook 07 is the next core modeling step; no service is built yet.

**Cost and reproducibility:** Both models ran locally with $0 API spend. In the initial run each fit took about 0.03 seconds and vectorizing/scoring all 188 validation texts took about 0.004 seconds per variant. These small single-run times are approximate, not production latency claims. The source report is `reports/context/20260929T114519998841Z.json`. It records input versions, the 732 training IDs, 94 tuning IDs, 94 check IDs, all tested cutoffs, scores and predictions. The title-only predictions and metrics match the saved Notebook 04 word model. No labels or split assignments changed, and no final-test examples were scored.
